In [1]:
import copernicusmarine
import pandas as pd
import numpy as np
import xarray as xr

/home/momir19/anaconda3/envs/copernicus/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# ── Parse your timestamps ─────────────────────────────────────────────────────
df = pd.read_csv("underwater_clarity_matrix.csv")

def parse_timestamp(filename):
    parts = filename.split("-")
    return pd.to_datetime(parts[0] + parts[1], format="%Y%m%d%H%M%S")

df["timestamp"] = df["Filename"].apply(parse_timestamp)
start = str(df["timestamp"].min().date())
end   = str(df["timestamp"].max().date())

# ── CORRECT dataset IDs (verified from Copernicus catalogue) ──────────────────
#
# Product:  MEDSEA_MULTIYEAR_PHY_006_004
# Coverage: 1987–2026, Mediterranean, 4.2km resolution, 141 depth levels
#
# Sub-datasets:
#   cmems_mod_med_phy-cur_my_4.2km_P1D-m   → uo, vo       (currents, daily)
#   cmems_mod_med_phy-tem_my_4.2km_P1D-m   → thetao       (temperature, daily)
#   cmems_mod_med_phy-sal_my_4.2km_P1D-m   → so           (salinity, daily)
#   cmems_mod_med_phy-ssh_my_4.2km_P1D-m   → zos          (sea surface height, daily)
#   cmems_mod_med_phy-mld_my_4.2km_P1D-m   → mlotst       (mixed layer depth, daily)

BBOX = dict(
    minimum_latitude=45.48,
    maximum_latitude=45.56,
    minimum_longitude=13.53,
    maximum_longitude=13.61,
)
DEPTH = dict(minimum_depth=18.0, maximum_depth=22.0)
TIME  = dict(start_datetime=start, end_datetime=end)


def pull_and_flatten(dataset_id, variables, with_depth=True):
    """Pull a single sub-dataset, extract Piran pixel, return daily DataFrame."""
    kwargs = {**BBOX, **TIME}
    if with_depth:
        kwargs.update(DEPTH)

    ds = copernicusmarine.open_dataset(
        dataset_id=dataset_id,
        variables=variables,
        **kwargs
    )
    point = ds.sel(latitude=45.52, longitude=13.57, method="nearest")
    if "depth" in point.dims:
        point = point.mean(dim="depth")   # average 18–22m slice

    df_out = point[variables].to_dataframe().reset_index()
    df_out["date"] = pd.to_datetime(df_out["time"]).dt.normalize()
    return df_out[["date"] + variables].dropna(how="all")


# ── Pull each variable ────────────────────────────────────────────────────────
# Changed _my_ to _anfc_ across all dataset IDs
df_cur = pull_and_flatten("cmems_mod_med_phy-cur_anfc_4.2km_P1D-m", ["uo", "vo"])
df_cur["current_speed"] = np.sqrt(df_cur["uo"]**2 + df_cur["vo"]**2)

print("Pulling temperature...")
df_tem = pull_and_flatten("cmems_mod_med_phy-tem_anfc_4.2km_P1D-m", ["thetao"])

print("Pulling salinity...")
df_sal = pull_and_flatten("cmems_mod_med_phy-sal_anfc_4.2km_P1D-m", ["so"])

print("Pulling sea surface height...")
df_ssh = pull_and_flatten("cmems_mod_med_phy-ssh_anfc_4.2km_P1D-m", ["zos"], with_depth=False)

print("Pulling mixed layer depth...")
df_mld = pull_and_flatten("cmems_mod_med_phy-mld_anfc_4.2km_P1D-m", ["mlotst"], with_depth=False)
# ── Merge all Copernicus variables ────────────────────────────────────────────

from functools import reduce
cop_dfs = [df_cur, df_tem, df_sal, df_ssh, df_mld]
df_cop  = reduce(lambda a, b: a.merge(b, on="date", how="outer"), cop_dfs)
df_cop  = df_cop.sort_values("date").reset_index(drop=True)
print(f"\nCopernicus merged: {df_cop.shape}")
print(df_cop.head())

# ── Join to your curated dataframe ───────────────────────────────────────────

df["date"] = df["timestamp"].dt.normalize()

df_final = pd.merge_asof(
    df.sort_values("date"),
    df_cop.sort_values("date"),
    on="date",
    direction="nearest",
    tolerance=pd.Timedelta("2D")
)

cop_cols = ["uo", "vo", "current_speed", "thetao", "so", "zos", "mlotst"]
df_final[cop_cols] = df_final[cop_cols].ffill().bfill()

df_final = df_final.drop(columns=["timestamp", "date", "Filename"], errors="ignore")
df_final.to_csv("xgb_training_data.csv", index=False)
print(f"\n✅ Done. Shape: {df_final.shape}")
print(df_final.head())

INFO - 2026-05-16T23:19:28Z - Downloading Copernicus Marine data requires a Copernicus Marine username and password, sign up for free at: https://data.marine.copernicus.eu/register


Pulling currents...
Copernicus Marine username:Copernicus Marine password:

INFO - 2026-05-16T23:19:39Z - Selected dataset version: "202511"
INFO - 2026-05-16T23:19:39Z - Selected dataset part: "default"
INFO - 2026-05-16T23:19:42Z - Downloading Copernicus Marine data requires a Copernicus Marine username and password, sign up for free at: https://data.marine.copernicus.eu/register


Pulling temperature...
Copernicus Marine username:Copernicus Marine password:

DatasetNotFound: cmems_mod_med_phy-tem_my_4.2km_P1D-m Please check that the dataset exists and the input datasetID is correct.